# FL-IDS Experiment 09 — Fair FedProx Retuning + High-Precision Threshold Validation

Single Kaggle AutoRun notebook. Part A: 4 held-out domains × 5 seeds × 4 FedProx candidates = **80 new trainings**, 30 rounds each. Grid: client LR {0.001, 0.003} × μ {0.001, 0.01}. Keep source-only pAUC@5% selection and disjoint calibration. The original FedAvg, FedAdam, FedYogi results are reused; no unnecessary retraining. Part B: 60 original non-FedProx models + 20 new FedProx models = **80 final models**, with 240 planned operating points (0.1%, 1%, 5%). Old FedProx is retained separately for paired comparison. Reconstruct thresholds, verify exact counts, report pAUC@1%/5%, score shift, low-tail resolution, and float32 rounding sensitivity.

**Kaggle inputs:** attach `FLIDS_Experiment08_Results.zip` from the original full experiment, plus the original NF-UQ-NIDS-v2 CSV(s) or the exact `flids_exp02_cache.npz` and `flids_exp02_fingerprints.npy` pair. Enable GPU; one GPU is used. No web download or invented data. Click Run All. Resume by attaching previous output containing `FLIDS_Experiment09_FedProx_Retune/` or the final result ZIP. Leave `SMOKE_TEST=False` for research.

The historical Experiment 03 prediction precision issue is not retroactively repaired. This notebook audits Experiment 08/09 scores only.

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, shutil, subprocess, time, base64, zlib
from IPython.display import FileLink, display
SMOKE_TEST = False
ORIGINAL_ZIP = ''  # optional explicit Kaggle input path
CACHE_DIR = ''     # optional explicit folder containing both original cache files
NF_UQ_DATA = ''    # optional exact raw CSV path or folder
RESUME = True
BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
WORK = BASE / ('flids_exp09_smoke' if SMOKE_TEST else 'flids_exp09_work')
WORK.mkdir(parents=True,exist_ok=True)
INPUT=Path('/kaggle/input')
print('Working directory:', WORK, '| smoke:', SMOKE_TEST, '| CUDA:', __import__('torch').cuda.is_available())

In [ ]:
PAYLOADS = {'engine02.py': 'eNqtfHtz40aS5//8FBhv7ACQSmiS/XAvaTiOLVFtnSVKFiV7vAwGAiJAChYIYACQEqXTffbLRxVQIKme3tudGLeAqkI9svLxy6ws/vDDD8OnLMyjZZiURrvbM4oyj2ZlvDGKdJXPwqP7MA6O0lVpjMLyNE4fjd+7xun50dnJ2LgLk9n90s8fnNYwWcRRcX+UJvGmbySpUfr5IiyPgnTpR4lRrpIoWRhpLsuN8MlfZnFYGFDJAxllDi2hldM6jdahsQzL+zQohDGHamgWhFkI/8AsZfvAL/0CurpP4wDmhy3xuyIM8SuY6dHp1bURrv145ZdRmjitH374oRUtszQvjbRozWE2Dzi+ZV5eXHmj2wvv5pfr4eBkbArz8mo4+nI+GG8VX/x6vlUCb8N/XF03Su1eWjhhso7yNJk8TF2zY6pxYfWZnxehmBVrce8X93F0J/4q0kSUsAUCaDAL7/zZg3iOsnkUh2Lur+Ly3k+COMxFDn/TZWuep0sj80v82JAdX8ErV8zSOA5nuOJCVR6nq6SE74OQugtgg9WEktUy2xh+YSSZKspgFCiA/2eBKivTfCb7p0fVc5JohU6SqPL5KqEp+DH2c8qNioc49PPEWaZBGHtFKKepvvmap6tsfL+az+NwnMVRufVViJxZrSlPZ56/mnnFLM1D4a/D3F+EXpaHs6iATrm8pZMPtsS/i0PLj2OvvM9DPyjcm3wV2q3ry8sbFylowcYBe8LeWebpOfC4B5vb7nrYADb73YO/WMThu8c0fwBONY1oToS3dmpsJ3yKirKwbCOMi9Aw3y2T8h2y7LvwKYMe43Tmx6Ztty5vb1zs/p0asBJHHDcsYP6F2YdWzvIhiHIL2AfqeOaCBvHSB7mO48HxL0PZ2zyOgsLjwWb+7D50kuzZbJ1cXgzORmN3Yo5Oj25H4z+ORl86H5GRT49u0tHRWXrDL1/Sm/rleDw8Oj47RqnvtjufzWnrdDi4ub0eYkfnH7zx9bF3dXmNreHtZHyj3q6uL28ujy/PseJHj97g8WzkffnzZkiCdntTPUPx1a83qlQ+np5f/uGd3F4Pbs4uR97F2fn52Xh4fDki6bs5vvJOzwdf8fn4/Gw4uvH0ovHw+vfhdaOo6ulspL/BgCjeMIObG5zsxeAf8un8cvR1CAuiecCcsNtfYHFbZfjp2RW+eOfDkeyhUYA0urkk4vACqnVjEdRgg62a6+HN9WA0vji7uRmeeBrZdioktZrlOnF3a+Qn2rwGv39FJXZ5+/WXKyKINrOdOtR42IV3e4VtOt3P1VhVFRZCXffjp906LIS6j53ubh0WYp/t7oc9nWIp1n7s1LW4x38AHYjso60C3t2zY1DzN39eDdXz2dXvH1TByWjs/XY7vP7TOztpvGr1wBAekPCP4TXy5c2Vd3x5cTGA7QLawvPJEOTiYnjzy+UJisVVmsZhcHF+hWJAz+fR4r78+uUCuQrFn+tOw2CwXlTPV3n6hC/T1vHpVxdVtUUmzZ20xYeueP+h/aPodN+Lbrv7aSrYvhauFGshraYrpyHmoV+u8rBwlbwKtp7eLI5IjbwXoHUT+OJ9W5BO8sIsnd0Xbkfc+eXs3iui59CFvRL8hUfNQaNkLmySiHPXabc7IoNZe8sVvHQEm3hvnuUwaap16L+PU7EEC79cLT1ADtEi8cAEe1Dvdtrttlj6T9D5Y+GB/vN4XTBLv4CpdLFeThzseRSQQffmYC3xwXW6MHkk7uJu6ZV5COvtwHp8sEkACdxvaFYH7KwpClCPS981b3/TUE7fCFJAMoA3VndFGZWrMjSGwSI8OgO1CMDDFHlUPHjQAoabmDdgUQCURPN5mO+iFDCneQi4yZB0F9QzTH8FFjLNF34SPfvSYALSidMNzrNA8QRMUEbzKAwM2McwX6dRbszApC0APYG5A3oADAsNsHHQAE0jtKzM4LvB7dX1MUrPNiSrEBjAoNi/AytlrAr4FNGaRGLCKIAh6IFMtlGZbEFQDlZcIPqiaTJaRPxn1FuktVHUXKx8gDEl0Kq8B7TmL0OFCQGuaZIBANAHBIP/GmTLjTApwiUYcCOdU7+I9cAa+mUIY9KcYfpydTRfnNfpKo4R5MDOzwwpDQAUgXp5lkeJ3Bgf4MTjUQogIvYzA5l5wztUhAWREfidvqxx6CxEmwOblMNqn2HkO9hnIBZQNA//uYpy3nsjA9GH4fpQCqsIoCFwwd2mRMovI0BQYYmgwcDB7wGTmFO7BbDiDSAyvrj8dWjaLmLKHigIZ5XBKGGlInY0wqTXnSoR7+7KMCjobwhed49cff6WEH9z6sMhYWOY9cSk+ZpTdwJbYD3bBoLxZ+Q9DTnv+XrqFAgMLSC9PW21AM8CB61DDxG0hYhYpHd/2T3CZPhqO495VIZeGT6VFjZyAsC7hQWtRETeBBBFomIXPB9bwMamAeyXa67K+dFn05aDzEMPdyfMud9ey3iMynsjBWbggc3c3P74qIgWUJjnKWhCE4Qz9pFrkvAR+cQ1TRvB8byXh8BbCXACTBL8AifngeY2D56k+ZJ5zCpt1dg0nb/SKLFmRLoZ+VFlDg0c0F7wsY3YdOZEhR8D91uyq5mfpEmEWp73GTtsGYXbGKKFn5qrpHg0qVs1pOKo9pRblGnCDUj3mFFa7m/fke3vVIN/0b4r28+KULaHFZqzaAaAltDn3q/ew1eyaJQmITNHEIHqWCM5YJl5moLBmzQBe5Rkq9K0BYNmfuPxNTYenXq3v3kng5sB8C9144CIhXlptYVyGyqu1RpPgezovyaBO5n2g8hfJCmYkhnMYtqijcPOcDnUKcwRB0a9g++VC9GbpUkZJasQ6pHX4HOsn2JjahgVHjqLytkAT7y0qCJfxOmdZR44wFbobBg0aIYjUkc4IqjODf01kMHdBqvbffAmCxcMmIVFdh9cVq/CFFiscIUNk0BDCUX4ic0dQmuyLe4daEHrxTzHFxBeKjRf/95syjzpmidsM2mfsQXu/w7nZk4CtgPYvCCC0Z5jRxqVHT9DXW0RiML1oojDogR0ulqCjgTLybMVZLLqlenLlHW8jmpFUs96bObdevqgHsFkJAuNTGSTrQa1jmjYSa/TJR4xaOP1UUlGqtGqNx6jRzyllmdlgjaH+gmfZmFWGkP6gxYYNEzY+x6ikJ6ilxAn9rlNM6vVK4BoKSCeEquNgzWmoL6QAwtXG0kSDnYuQDiAKImHYiYUHu4vrYRGmqXLOzTh7mRNDdZVNQtFpaHWoIF4850CoENZoCa2zGS++mcCKmLdRdtA6k/2qFSFeofVQQMGMyGJJiIfYme7JSd3z5Ep2jmahHgIN27sL+8C31j3cCE4DxsViwQd7ls8ymItkQnupHxMWPZx8J6ajb6ptk0Tc/wAtos+Yd2MbKvaAxT4oNanyqgRU86n/zZWQ6buaY0er5HaVWZlQkZo3wY0eliDUrdrhVsNzMZykwC0Aw4ApfSEzEzaN5Nxibo2mcvwxPjy9OaPwfXQI6f6cnT+J+mrPn7wHVEPIweLm2QOR8ccac49KLU+dmxpqDOHTLX5uMf+wv48umh4CSyg4e0/yuf0sZLZw0mlvgZl6c8e0CuUNJkq1RpIvYBECgl2IjqTNop4hZptKO7YFh0uorKI7ADieqv7qS0rDN+FdTgLf7n0ra7oCNz7SovYfR94EJHXYXDw/uPh5uADlHSmbpdKOh14+zA9dDcHVsf5ACVOFz/5SEXOJ3j8xI+feTBt1ZO5+bLuOZ/mr2bNDP70cLKplo9csZHhrS+EBU25+kPzCKQQiSIZJGPOALcEN5L4YUuxhJVz5oG1m0cLqVoALdrb5vhNQJy5e3ivLw0wSFTTtiED4+xZm7o1VmgpK0wVuBHgXMAqT0HDjdLyFEuHqC0VMhgNbygU9HvXG13eeKeXt6OTnkFkuic3R2keA4Nuvx2NYP5AIeRytG0+uCgU5YbaddcgRYrk/pUgijEIAuMM9W4fg9vGPhXsNDxTNJY4/btQ810DxxhDVY1OjDIFv4d8fuN4/DsBAlo4QHNobX5TsLrtH3+0+5VD6mqRZQvhh92HqSauDDzDJkQJWQOthJbpsctKeAglEPzcOKCYgxdH4B5ZnTY4F/SfUs0ItnEPK+2EGwTm1H1BZduLWJgEvjSlkIzkKzSOsTX8hwxlKaGu3hmbkDUNGg2VsIMINcqlOJAKiJ7AkOzaAZhzZQoWaRq4d4C02qh3SDdhNaun/w8nQtytMOaAX2Bk7AD/qRWbwc6E2/Ar+qTl+dVWOgmknjQQlfYUJkF9AzX23wgsEQl7MPlDt9PXUOnbJhC+nSAZp+SQ4JOO2Fh7ENXUgE3bKDVnT7LPBNyShyR9TGT35nRnIozTNLiAM4hpBtWaCFLRyU8blDi3hQd2aAHib2hvAEzFzdYdaOQr3Y/9z6J0VdAHHfkBqJNqrvLBA9H38w0juT0zrvA3Bk4KFDg/z/0NqOA49UuaP0x/MpsyDbV3Bw/rMgX7207tCyoTMRVBuclC7JR6e99lKigNB+VRMQeHHuQDR7cdUEagn6s1JKiMsRoDbysWWOTAgBeiZpz4iVemHrqaWATSl7htR2QpQOE5PiXhgp94/G2YupenEHFJ4CM2rFEmUEYDJ271yttahkuXx0ZshkTyK7bz97KdafJcQH7cSpNpPTLzQ639E8Us3oyTmNMetFLIDWdi19zA5P7LxUAHmnL4Gy7CvADrn9j1Zvz1kxoM+5r8NXWxH6pGhXHIDEYqplabb2B5DFBI8I3zdbEDAewK3LEMAy/NPcWZMjCG6ojmQiExyxwfD0aj4YkpKsUlqA/F0aBz/EDM41VxrzDYE8LoDbm5+A9LSeXmql0kEbAFzkrD1xXxHVwzOLwEfvA7X0jv+LHoPanV+nZ/o565w34FmyVMluOr0rI26U89tuXXYIYAbigzfun9PgBUAVD09vp46F1f/jE2aVUojQXL49OuLMGKtQabugGQ8TNMS68N1Kz0Qi4h72ftvgDm4gW8/p8XWtmr2cMNsYCArivX9ndr47q8bnCAKLbD0LMCnlJpUmmtvRBtgvVbIteiEUbY+K+B1d8QWLG8fASlCKKP55vu5IF6fxBrdtLXauuwx/VP1SDTHaCHQVGPoIu/CHX/MaMzEuZYlIUnW+hCxv0VYHbXVQSy4BA2CSRHLKuBmf9d3HI558oxB/4vQBpiV1bY0qvit738wRT1js8H47F3fPn78HrwdeidDs7OhyeGeYhSp3UG+4uLfoaZLAHyFuDDWXQ0K57cJ7FxgU/cQEhmqDjg30Ab+rNyJ1hNwoJBagrPU4AdT+eNAwpTH/DerzIwRgByDBnDdlqNILfOchOZduDcxf5D2L2zqI4U72IF5oxboc13ypQC1RbsRLQIweeiw6BO13buwycusmwdPDxN6+Vbu+fQ+pScJNuYQi8hKYhjPr5fYDJA4bb7szwtVNyjKm0ZqyT6pwB1hCvD51Vo6X0JdjxQ0YGuDZWWkt/raBUQnUSWUfAkHpqQET63e/wRRaof7Cl547jhwQQ+4MjFnimiYCIXr+2fO7UDxZUOm1JrT3xFHhd5/iqISl060jwCAwq4SooB6gof1MSdVBFr3gfLF3c2CyXaRyWVr4JFRRcieZ7GYRvdqgiF1Wlg+WKr9mQBC/QwMGQCq8wnZm1rzCmzA4FzrU9bTqAp37xxygo12IOaYD3wnmJu4KIy9/UejupWjU3QelIbsmeTtJ6B/0FxmzM0GIHBc+UToPkqjo/UYV4IX5iCziA8/SjINbeSkg7xfA2Eec+JEzMBpkMhpfTkp3f1wVmfxi6Ba9URFCx1TZpcP4Dac7yLU2fvIYnQ4BugHPXYE4NRg9RlXz9ARGCS8BEkqpgYvDRAeWrbJeFAchAN4qGrSS6rdgQnE4+QfvRNQd4NmYflMk3Y91UHrOjo8rEruVpLH2YL6qSOjGPeme4BSESAB0dYrJ8iySg2qHxRhm61OxFMAW1yQ8kI7Ffgh5rjA3sNSgq7nVgmbYUp8DDIMusdMaF7LAF9WEJtaPOw9cwbYEwmpbnVaAhXA4FDuTReQhwMo9qC3Q4CiKg6QaQ2oFqKqeu2bVuZiT21HahFdnjyWBW7UiNTp41tA8exL8+LtGB0QP3Y+L+WkQUO+renOYA9q1oTmgAPXEdWTkxZVUkxOp6Aewq4G/WrxI+MIK6uh1eDa4SRUtxNlFyhmJFVETreawG8RMfI2FDfWfsAS1T4rIE45flOtWIVwVTWrbj3ux8/Ec1gyZZmAKF1A6h9+mBrpq5h2zieuctQmcZGwAZljph37eO/pQSIfgmu03OYpwjdsLW0MjswraeHh7mpfuADs93TYRUOViZtntFu9lmlalaR62UMWqlLIOBPnfY+qDM3YUq3p6dnx5RNdTs6++126H29vry9GhsKnZLjBJbGpTjCTt6elXhEM0wiKSvYwPv6ZhKHOZXpjV5R+ijGSFt53Is7xlwv5HJwAmXuwPDkcgXFxJ/asAN6yR2CEdgabetBSWztPO6a1gBUyFaDFqAtPOqr3GUMXxWg5zqBRToHJgnLI926v0kZcpOqJxa+Mp/aLh06KP4+esHNf6WuZJsQ23AxIsSRwobb5kThPnB10c4ZB0zoA8lWBaipjZHOZqu8L7NEhErIuI+C0Gn9tzVhdS5RSgVlg+/Q3c9iF2fj8dnoq/flbDS4/pNhtSEX/4IbD39wLsRoUq6VfmfBh6klBZo3K/cfUQT/zTjFKJJQacvqFMooQLeCyYjTRf0RZSgjm0V0UKXlJVPKxjwqMV665f7BQFsOIHGHOrzPnFkcZaRvYERQdwfwCKN2qMy/KyzV5EkcdcJOV+A/oHvh7bPofLZhIOzd0kM1vFaVQ2NRHkwhE6woiE4F+ixlkz0TlQ4wN2CehXVvB4G4WoWB9vlCXwYn3pfh6OzriH1mTqAcH18C/MD9wiQXEjrSu7K/fmXBYELQCy+BtLtsQUYoWqelSx1Mjh7ofPuBfU9Zhud2pctxMYw6gbT7cwwn05eCZA/TJmRmVd0S04jkSD+7pfT4uNHPPJfDTnjU2cuvJuZ1j3+5PMdEy98H12cDUIxfrsE9Hun8Kbg7eQYGTIwkRLvE0XMsgdd+HU9XJZw7DWYVvvaahegKADpW6cpWx24RcDMuzq+sJHEu0mAVh8gJOKjn4R56Hh4OzkVi94pVhqF2p6rAIFo8B7qVLnw9DsEQwEL9GPs6j5LQz61EALcIeL8Oz28tepI1nz4I4Mm9Ve+Bm5GoOAsQsEc/D3gST7Z2OEkDg2Q4BYwbPofWUUeyOGcekr6xCF/6sQBPWaBLRsCpKEPE6+EabDAlGzLkEJjfgFQjk4GhPaQFJX+5SKInhAFZiKeVfSoFifQDbu5J94ZGU9UlWFcapCqgOSFbZ6XLmwJP0dIZBP7S4jaIgJch6nwgSpyzpYtzPgB78wiFDFxfLuAxxDQrQL+P0AaPjSxbmwspZ3Kw5XQrD1KCZa8+OyRKUQCN8jfp+HB2n0I3FlETTfEykjCrTvEE88uA9Ak0kozxuqrkp53GaH+f7iRB/MIDowvibj1NqNFU7pSraLnZbbrZ21QqLm6sRayB5g5iH/DcfEKvFG2G/VaePbBQAW6QI2Pt7O1hdl6abTz0QzzQxQBJeMusJ/CSN3cSFSE/EZgH/A5+z4bcWis7WhCn0oElB9pkbsACqf0MmnzP9jf3kwO8NDX855DoKHNlzemB8/FADikX4ODdD5Ieu49Lxs20ah3z8tBb1/wxy1b4bwxUkFOT4TielsbkdhUHaLX+l7yqIUlpq4NZbOgxcehLkD/kF7mDKFW7ErUjTfSIFTXvOuhgouJhU6Xcu9n9KqFgizru8BTUe8KERKsjpL2YhVFsKUZ896H9H5/QYdEQB/UEuo7tmor4svqMFss0Cqyltcuo9Nk2o9qKqnQthgI0mn1PE4wVJL5uJps2Gw0tEXSOjhLlYlv8h9RZIEidYGySgLGQTiFmZ2qUBkziovlxQHvPOWRDJ6boR6/wdI3873afRB3+svZE2jN9gTA8qOtuZ56TG423alRap7pOcw61x2haMNU4h1asRJvFAOxh1tHSL/EwktVcIz0UtAiaqzj015gr+pFyS4MwK+/dj4Ku8WAAFuMFrtP+KChaex/hiS9HcvGTPFx4nGcD7gOg2ru0iEAmjzoi8f5K79Cp2HUU1IQdIDxsLpF5Cj4DPwD3rV0prJLTQQjvfGgJWzG1Jz3RmQI532iDW1U3QprvXXqrOvCprD+p+F3BwVRh93uF+Q1BxlRA0DscaXxEhgXJ1SzFsqGXkEAq5Z+JMgkkcZQzOX3jFAG4ydpuC4yfbMj6VFLInQPmf4+wEi3M0s9QcIWqesvlVDd3ZJB9cH5+eazu6BDCJHcTCS8D/didO9nVEjhoFVWhLdrOdwbD9W7Xltn2AW+odv9BBjKlMySXINe7JVx4W4PMR+HixlZhpdoms4RruT4zAjQk2c2A89YqVRYQmo8aIRWkTFiVIIsd5gfv2+3D2cGP/1H3q/SJ7ADZ9tCtamm6a3dL70O/qJqU1QaB2G1Aeqvum47IK4JU11po3nQ4TCERFdhTq5dkrdeteITXWxPym5TcSxaMFewjikbVyWz6JnWazag6+1d0yr6DTkwNZZ6ytc1UqQpKbqSp8kY8UTL9rBnXc9U5RDFpT6c2s0SlSytR8Fd5OnNVNoF2bRJAGOk2AcNjsyxXzd64T9n4gMFNQUcw5F/R6bv/FIFNIuZR5bRSVdHSzsW/T2BI/xEjobwqRvo+TpIfV8fF/z12Yc/Dqthdu65lq7N8mmM1ntwCu/9tFlPwX3fksR8Jh2lP7L5s9c6VDwxKpcRw9hOWoImY4Nv0gDfzkc0ClotHhV2ZMKICqtREJU0Vr9RrA3Acut0DXdMfsP05+PA/pUtqpFXRYL0byBB6fbmnnuRFSkIImC0o9oGpIwBZgjw5oHPusVlnL7Oxao/Cwm6jTF6UU/ZUk1kVsfGLh8LKojgtZWgcPXQJG2REzpxiGjK5H9iOwxzS5raMQjaWcfA9TfVqAOdFdeNQu2VYXy3c/lreD6yIPsGDQzFjLvCRBUCu8PmO2KGoM4uWhbx4g7eWgJK4m4KfvSgQCT+S/GEwzaUomC8PwO1+7j+6WD4xn8xp/8ltBvL6G1m5MTGRhJ8DeJ5nVUffcayMoSQQP0pUwL2QKYH02FAWvFG4BJBmIg8IMpIq+ne1ENetFocIj/PwMM2FHYe5OVsFfu+lavTv7GpgqcNtmOkAs1HGrFYdFZ6/9qOYbqWri+IABE15Hv0k6ByJvQRWOnZz/nKVMo8Y86Xm5gsu59XzXrQcOOzGxjIKrsJf7u3VxNxWugnpUfoQHfPwRbXCfAd9QZevfAzdR/HVWklpxp8ZAED/WLema+aEk7Se6+vwGNyuuqqvuMiTo+vh+PZi6I1/PbsyMeNeP/LRk8NUzty3Tv0yUVFPWirQ7EhH333bPePumORWl9Cezh2a7LGNAvioW3nOPkQrOe0n4QLUxRrmtp5swNHAA70pp2TS3VMNEcnoszrX1G7QytNN4hwqdenCLZERtbEaROWo7b2bpzpRIzegRT2Yy48CVdmqcE3QUqv5PJoRhpY96scAiVboNiZj21spjvdI2JjGqKLZqrWKZzPYykAL/4yt+mWmn3pm2d8524sOPm3QCDKNaX+bNrVpnKr+3+0uyjer28qc/9cIJn9sBI/Nq2W6cu3VeRd8xqTgszF8V2t5xz5NmRyqEnX+64FQYspSmak22eE8sUWFz5o121/PO2C8qxb4WLWhboDUZQYSU80E5uCWiQACzmWWImsFt3m6nTZOtyUukhJFckPWYessevJcX/fUnMznv1F/U2Arst7EUXj2pDMdFeLxEzTCI29qE9Ir78ruYXq9pXUrJkuxpxkeqvOZ5b2vDtRxAG0HtRqcydvomi+KFd+LrbXmBweopgCfoJrHkEmWUlKMZEfakFqJ8uWY77kpY+zNTat64pXjiTOoQHrmm+GcbijpwnG8otxzXqVTCXsBXaqVyL5IFTboKXtc7+nRrvXlf22t/TqxSvtcMBdL+pHFuRmMf/UufzXFFpAB62u+44wEaWVtsW2NhTm4vb48NjmEZ3HnE5M4wpyKD82MhT1ZzziPZbFwq9/9cRAK+SUAm5ll9xlt7E8s+Zbo8VU+6NjeWSnnLJq7S4HWk6NOp93uTZuzbmaoVfhCUqoGPq/eHLAMZiPJvDWePR+18fP+0+AXpLFs/Gpw9wyhDewxDPqgKYosxHQqOYLxv8eXePhGQHSBSIiumNHdWLr9ajWQjO3wPViy4Z53wBNkpLfdsvm7PTKcGstb3oW8B8G4kQhk71/U1dn55Y33dXAz9M5Gx5cXV+fDm6HxUnf0+u5luyc67ibQPKFLPQh3wY2gext8hV26afUFXp4eBuO2D3Lzig/t2rvGzvef6vJ88WyXWXo3NZGJh8TW8xKV6cswtyFgKYq9fCVv19KAAn8jwzXH6bx8xETWeegX0V0EgG1Dm0dJrY5xRe4JawNKeMUlUdIrzn/rFyi0M3v+cTGH8sgkq2vUvxqMxyoHiWezk0XkLxZ5WHGRZIXqpCLTMtX3sxVlxtVc1ZP1Sm7f3EuYL157o0uoNS+ww2irTJ4X3Ekp9Zgjw04fP7HoNnZYjv3a2kr3+Zurhnor60cxqUdR2OtbsMovWgfAr6oH4lN1N4jH+xf9mye3V+dnxwP86aGqe1NmRN3vS/Pbm+OHnle+s1ZyNTQaue49/xqKIpXrUuS2mrKSYukE2/QTKPD+Al8o268nsubFK3zR0Zs14cCexnupcHr2DyAA33K91rMn6myCEp1B9C9BCDiLEl+q0L362QBt+UgZurn/8tDLJw88FfJza5oIpoRQHCNMCbLoScf0JgMreKgoIbZXK6Rmwb8Aa8zpq9w1gnRAnW3corYvfXRfDg5wsoBwnl/7tFglJphxLnfpGXaO9ArunYaqaVufG9QH+GxOf37WfSZz+o1cDo3op1fXdUIHnTFUNFdzgtn+j1J1+ir29tgIV0HzrcAUlOwJSkFvr+re8UxSvhGZArLLp70QoqFXCErUykXiCU3HwMRnlKm7dBu5orSHdn+5bKSKatd5lQu1ky7ab/RT0b6Zc9pc9yx9I+9U70kFKBv96IR5e0oto2I2d7mcLPnwDeybzoVTZ5bi4XBLXdVwJ2bDS1PCg7wozArmV5cHwR3blSGwt6vl0s83bjWSQ6l9dxtLZ7mK13SGtydyKlMHjJmFu+bjeEUZEO+ACEAjBy9aceL6E05fDuion8OYmJ7MEsaju6LMxQyvGNEdQ6iytZ+aaX5ZT76ZL8xlHk4GGDnYR2/8waxar2nxLhkU4Atg3yaJlMIGQShBAEeCnSydetmKfkpw+Fcl+XrnNzaxuVn1RX7ujT9/44fctGM/9RNR3F7/gTjt9+C2orjqIKM6ZuM0IFqZQ385rV/rulHbIwp/jw+xG9dg4rj8Z8vHUAO66kEAPsPQGvm4agp+OVH+BXUyPdJqqi9l3W4iOs29KciZH6G7iu6rn0cArfaKMToKFVfNkSj4C5ry13CU73Kw5bHYPXxXKcQabJtvwzb8fQ1outfCXA+P8TQbHS28kYFp8OMhG5gtRA0o2WP9sg9R4wrjsAzBkZLIyiOgSdhaFQnVSq/UcdluDAFrl0tbVDKlV1SFeDmnCk5vfXq01ZLub5OB89gNYIcc83mBzVTMBVzaeBXISsrRlD/PRwF9wnqyBJUhXYyXP4Lr/GeU4W8vWLJe4K94VHVnV97J8PQcsaVQQQ26K+PyNfhnhSkrNlC/iiRlCzez/gml3jP/CIY1F7jtMYUovTKlcwfNxTg9Gw3Ovf/EOHU1LTbL6p1sB6aOcQK8wP0u6t3c9UPowjz/Vor6nWBnkC9WaEav8A2TgDK8gOb5stgyj47QMzIF5/ihEpe/soGmh2LWwkTfhm6sEzSp3B10Ivin6uSm7OmcRcUUdMwWUVoB/1pa++3WxZ7mmMXnu5lDq8JvCvk7G76jAuxy2r3qV0JanF1QtaA599hR266rF9XT3DmZkiOPqrix8B1+rx4K+tk6z8PTT8+DzjwWJs/s8Za0/h8pDk1/', 'engine08_original.py': 'eNq9XHlz20ay/1+fAnHqLQAJgijZzibUIlWKRTmq1bWSvMdjsVAQMCQRkQACgLKO9X729+vuGQCkKFu7VftcKRGYo6en7+4Z5M2bN0cnx4dX1uC+UGU6V1lt9X7sW1W+KGO1nWezB2scpaV1pJLzorbqRZZmEyvKEivOsztVTlQWKytaJGntbxxhlpma5PMozaxpPkvyRV3tWzEmpUlUq+1KzVRcp3nGcOppqSoath1Hs/SmjKSnVFaSVr/laQbAZ7k1VQzIwK3iHNN4WJzPi0WtEmuR1enMipaQ/5xmmSqttLJkVZX4G2/evNlI50Ve1lZebYzz0rq1ANKxz08vwrNPp+H1r5eDg8Mr27PPLwZnv5wcXK00n/75ZKUFb4O/X1wutbr9DQv/8spX2V1a5tnwdhTYu7ZZPK7uPKt6qDzrtyrPPGsaVVNQwLNK0CWfe9ZjWozTmfKwzUkRlRWe6jKK1U0U3+IR7NoYl/ncKqKaJloa7gVezRrZYl48WFFlZYVpKgAdDfivSExbnZfxVIDxowGVZZ1GP8tM+3iRMQejGcE5kkHV7UxFZebPVV2mcWXGlnkcRos4ZJZhL5CaaKLColRxWgGGdGxsnA6ufz0/vAqGNoTt4G5yenIBwuL5oszvm5eDJJo3L//IJym9jDauBgOa2vPe7Xlv3/X+6O3uvfX2ens/jDa+t1gugVM0w54hDUBwmieetahImEl+ojK6mSnrZpFMVG05VTSHXM1SUoeqVgVYVOaLLMFvVczSunJ9Ais6MSvDSiQ5WhCwmzRjGQYBshoQmEwQRXWPt8S6UQ85i72yQC2sUUJwt6FYkGBVR2EV9Pxeb9ff+HBwdnh8eHA9uAqeNqwOUfrDJI1rR/ALZ2Vw71WqBFnpedfvefMFYHhAh0DtuhaJ+D2J+JBA9956vAL/lefe7mjkyRqG2M8WkTnPFrp/YZ12DfrTwjf8+yb8qLONWmBHXs3QnRa26zl6CfP8fqV9123WNuLy/7z2l42NRI1hziaqqp3I7ZeqXpSZ0Xe/mkZ7739wssKvoC/0G1VRWUYPTuQl9UOhAjRBln5457p+nd881Kpy8DhV9xqmywvACoZkSZzCy91+4UOqsT9/fpukpSMvVXBdLpSn7tOqDvNbfnP3C/9zmdYqrCGiDkHwE5iNysm9NEswK9jzIMH55zCLsuAIWqQ8rBctZnVQ1aUrq8/yKAnnebKYKYcMElk/bQHkh7a6gIXegA4Fy01+Vag4JCMSksELZ3nMKuTY41maVKG6L3p7IdjATz/aHpaVRdz9+SoswUGgEVynKtx9mFndUQ2rws+g4KNgvo9HwluVvroHBhr9ubuvWTQX1lVKJSFI4FS0KzHPPjU6YAsa3X1waF2z2M15lC2iWbimp1J1CBMdkhOMksrZdUG0sTa38SKJ/LQKo7sonZF9ctx+p6cDlXHTkAVj9sjhHfWTwXLuIu/BGxde4tF42sX31qGqVTlPM8hCGlsTWLiC7OENXL6KsH1lVekk46eKjCUMlPap9ZRcMLtZC2uQWwe3/A0BUgVPXzZYadKEzcFdRJ7wVj2QuDjjYoj2kbsvg4kGWpocDPGGI9ePigKmkneEodgTkArF8wdPj/3hiKE/WoxFCafuAIiTDO+ikevqtQkWZIfG6IUg4nMoDuGiYRE6CbCphr3RCKuAKFdmyzDlixlMMqmKFcVlXlV6WtUXiNZ8ga4shyfn3iau2elEMiAKcXSmMo0jVgOSP+/2DRJ2GPL8MLQxttloly7S0iWNo7dHtCnS+DYYjvaxKn5k93oGb7lDpha6JgZTY4Vc3MWxyyKg8Is1SazOiskiLLbsf9pbREgWLLyJiroIeuI8UWSojJUa9n8cefZNOrGJ1pZDiBNxFv+zFwQ9S8GwIEycYSq7Sof2p7fXMYp6K9TsrtpHpsHzsQR0dSgrGjGPWMPALJDBNNCMfhmlwOiS4sq5GpRlXjr24PTi+h/h9aezQXh+GX44ODn+5fLg+vj8zBaIxGRIOAEcuX/Qb4A2Wg+OAQFKeHR89nFweXF5fHYdnv91cHlycNGB+KABfhc89bzdL4SpNDNk3fr1BT4gjL0KT4+vrrCSBk0uZzHXYMAC90/O2x51IGZFIISw1bE5PwgR3MICX52e/3lguwFFscKuXQQP63fWIU34y+Ds+ONZeH1+DhAHJ9gtaNcLd8OLweWHwdk1oaMtLu3TAzpixhDuplmYZ7DJ3j0smHbZEAy2Y56O5OLxxNMBWqLu0ljJG/zIhAS8Nd8kohsWzLyaBXMfsYBzT7JcqOEuDFJVIz0Jnm77d36CUCyeOq4fFwv6OyMcJBK49e5ISxiIz1NCDidco1NfaAVKpRZzAiYm+1FBx8NZequcuyU4DKEztVLIrJL/ZGL6qKqu8A9JkFOZk9KMlnojrQ1jeL/a3f+s0skUwQGD2OG/LBigVRwVZAScuf/h6OPQ1iCEvOizQbabqI6nIc3qjmxbbTKteiJH08FwHt07ux6NBr6xSmcO/JDD+N67YH/h7rTzXdfd7ACm4GAWqiKPpxVAy/buV/YHs4GYfo4tYRvw94u5mmn+FS33eAx5QbKEcMk1ALOhC3pwORCBXY/+7iEW/Anh3E8/QUyzSdA6e22gQ7SycG29Q3D4R2xX/C8wocFImn9fKN5qniGyUfAwyhlikh9Pc4irwwJNBCQyvP3xncekIPPulaqYIeeTuEvvVux0Z7tus+R9IGLDRpuzP+d+2GAzogDSER1x96X5Qc+IKgSAGSwm7EE7QQuKBkri8nZPK1mg4YjHKZHUACkQZqIcUUZ2LiymFfmnGZwcP2mXE6fG4yjiT0lEafck2TMze42qcjvHbmFHBfnRdLYb9dmMOOx08gI2CpQu9Zb43aesxJFpXZnwkAvAtjRiPytZli2rfEkIOAhjRwh6bL7t9bbidPOPP8kkNQ6GRce0dIwKC+VzBIhU3E9605K3q0zDOB3pSsNNUK6XqFaX1giWaflTR+MY3P1aWbpZlqGHdeJz8zqx6bKD7Vw4KSOO5cI6DzNQRxIUHknSExz5lF2XDzpiAhHKvHgIKSIlY4/MXLjo3MOMPDzITPgz8RPwW90MlzsFLv3Z6vnvNxlVh3k+X4DZm2Q/nGL7d9evfl8ghwLf2DIK17zfiSmPabFOeMBvt8GAAguhRlqNEXND1mlR4zuPaF1E3xdU8NIe9Oz8DFHB8TX898kxvGR4cn51ZbfU8KkO9DkqE8fdb+lIQiGiLppnIsZXujXZx1q35moNNhCFVowJDYEJpWWjuwk5LyKSDPjsbkbDWwnZP3uRIZj2OZ5g6WokGsf2haJ0NasjAgaYgLDNHQbUmqEQmXIuPs2h9R/9JF9w3rTCPI1vm0XwdB+pzIJT681N/z3tJUMgqoOCLxsdQZJy4QtlKlHGdiqQRwNFS32jzS3uIoSTgBHA1vZN5EC1QvZAm52WLWd3mxvdzck60W6KKxJEGBh7m827gbAHCA1RGBQj2M5rnrbXzdjU6SuSJacd2fYLSGRTVEhDQjBL47QGlW/SSNfcsjspjTE9bvJ6asppsp/KZwjzadDZ/o7Z/uamA/O6tQunfTcNWgzMgL12AAl5NC9CMoM9QathDe9TP2x1NL8pBJEBmE93nDts9fcSutAdVUdkIFx2capYlT2nu0ojue5rJLKVjkYQtQGh5MVYj2ZRzluat5931Q9fsSmHx8gtPpIxuRpc4jG8uh5woiELBg3WaOnEQ1vB3ianRa173pQAa/MdRs5zoJSXa/y07nnJU5vujq9W0EIWSTLqpvCsPYPbl8CEbbbQ+hUeQS/i6ADJ9XTcY4j+AoHbpV5npA8+HR53bHSTgRhzyfvm1oBl0yQzs6iSZcK5ijK9LSBCb45YXHhrqbaEkhB1qNA+GngdU9g+muLmoqAjGOltZMaL8/kcQaoU3XQE3OG+26ZnzDev08dBPIlwNyZxN3UAqLM4c8QDpZhgjTvkZ8R+GD80BGsKn824JmlnslAFdP820HkDmsABgsSyiTnsgeqWhCTN0RjumDqH2/Ac4POtZK7csjvyuCAw7myx9pZZgIE/B7UprMnhhpxZVM5D7VUxVzolT2+D/QfM+FpefvDpQ3g5+Mun48vBVfjL+fWvkqAPrjq5MEtMtCjzWG9p6ShFL07iXJRmxAunK83YgqaPi7L3/mWQHrIzGkNl8feu2bhUt8IbquSU+edKaohXRI169tA9dOuTlGQJAhMEJAklYnUazazL8w8Wdm3B5utjRO4hkdtHB3XflCq6raw6VWT/Jzn82rCU7IKDXyxLlC5hoiGHC6R/8Ht0fjNDzGXzIUxXgzFOVgrbbcNkj5rSD62wljtn5+FfD06OD8PmEKbDFl1UoskeFTXhX26SyCr7zvb6Fb1OO7MTTWhpTkTJjQx7I6FzucjCMenK3Pucl7ce6U232FGZSkc1z29Zh+4BIyIdISNLWfLBh18H7v49zDHMRVaBfnOHBw3te0rYHwL99mCP9hPzkuBlXHTgXJ6fX+8sV+CxA1UWJRSw8rPigYhSl95d5MF1zH0uNYcpdhWTbngJVZwb/GnsIlNU3Am+UZ425QIki3U5TPAHjH7sVH3n/uH56cHx2RVx8vG7gBYBW6k8CdPFesOWMtClx7p0vbZQbVrvItcTtjTjgCDVH5ryrengCiLMXd2MVGQy8kUdEJd2TJmsOUbv/RheqgppYWXvQxfzMqkCjN6x9UvT+u3zmQ1LJ0421f3trx4QiG2zEd7bkpa3AasWH4o4Eb9UQXu+OJSukVELliyB9HzQsL9HWakcp3aSeSPMVIxbyemxmgTFpC1j+2nuk9cBkWEeHGKd+yUMn4jz9Cvr4Cl+aoD2e3vJFyqP81l3oEm3M7YZDUwC5C8+HV3ZUsULvzauqO39eAF/uDpKGvWouLrjFUEUGuYzU5AdsJlpF1lub8E27foWQMAHa6RaFR9d+XTiI0duVHticppQIXf3WckcG97h0+kgvL48PjixydrAOS2qqT64Q9iLeGRBwRqMZhyQBfMhgeMwho0gv+fKRoVPYNqDYCNunNy+JxW59YXWb1VZGbYE1eIQYQTgeUj/Qq6wSZwHiKz5OsSD22ivgCwQ3pijJBrER0wS+6s6WPG2Qxox8uR1KbXumn1MGtpLFv8FG28it+tPZ8dnH8PTAcj8QSfYuVgRfX8laKxY0KFIYAhjxDToaEGejdMJVY888VUdR+VJ5cHa3HwybsHeQnq+VF5WdZN4a6oHmvhNrq+rqXJDYV0MxwzWi3UDSHFEJnDkCDRgXlJEBOe6PtaE16LIYE0kSeXcxyZv6vTYHZstwmLQEeCSuVXrBHcbAu2JR9FmXX6EPTr/RLijRaxVSOhO4h/CpR1RLcaRZZFhhFBop9VPPtu+12XV/fbsnLpyWUTrICtfeP5n0T+7QBADqRKmPBc2773r2U1Mb4a9irqY2dFuQkHdxwo58YB/+F5UZan+f0tAx3AjKrE9RQoCs1iUDhj2Ig9Waba/RK+jg+OTwaHQzIBa2d2q0eOkj+5rBd1gU0ZRJ/YJiVZdgz3NK5WR03iNSxEHsUGMVaRO/ymYrHi0JWfUGC37AIbemv9lS/6347OzwaXtmZl0E+Elmy4puYg6x2TP/dmrXakQdvimEYA3I3aq5Ayh1kVz5yLg4MHT5bmQzHHDseque770kq1HrGRM/fP8jbf1Gii1+iqQUoIrOlcwyWJQ3Q2bo0wTl9RRSVe6zOWn5kZS7/2or727Th4ZhvsnOtHU17loZiCXg0Tn9KJLGb0M43xJHht1SrNqMR6nMefFNypLJ1nYCS1td9mBx4uaYlQG9DxhFtjQMZj7n2Gsa6q8wyEqcexySOcUf3DQHOy6uqzkIqLPlrp6bVete5x/Fe6zzvGzzhboxr9NCirqLMBXGJdmb7QJU9bokIWnCx2Mr4jqOopvQ0h/NJsFdbEjx4d1sTXOJCrnSdiq7sHTVo2uJgtenlTQrAJtGEizPFCizjxsGfDY0Nz89u9YV7EerZHVutZN8JpBYnWbEfxqj2Sn3RSxGbKSNz5LL1cHLvkipo2+j/k8mKJQqtYeOROHxKdBlC3hHbGWvHJStCS7gbkjQeMoK+JxPKoVdBGg9nKBlh5Zb41XWRvCtJR61of9rYZCZvRKO+Xb2l7oX0GCLU5obgZz0RDcDrXmcyrWiOuSjJqryXLLTOeQPJ6pG4rF4lsxtc8vLFdU6ULI8hhSKAi4Fd1jIW8hjKLjYqqu0TPyOSRgouHSoiNsAERwAYhOxyB2U1YGQzaYt9jtaGASR5amGNh362BrwWLsKIzX7y2GFJTT9pqAQDs3D3rE7eIBxfVxILVa1PA6uuKtUx9b2xZWA5A/mmQ5XVsz8RVWggZ0pN1GpGV05t1y5CEX5CaTkmyrcpoqi2Q7VVNnaQosr87ylzP8DUuUPhh2U7+lvK89+8UkfzLLb5wmGNmUcIXKVQoLkoSymjGq7qbcKFPmUSMt93pYOxmM+11gJq9Lgsb24O8Xgw/Xg0PryYz7Ygmjrjzr4/m19dQB9oWSowhiW8otqScqsWkbKTUtwkieBCFzQYPrdwLlixs0OG1YszST2wFNQUEyVn5iI0FVv+Ze2wqtdBhkSNXXtvtFeq/k2U8rSRfmmqRLKsWUWvJZnzbUnt21WzZdu8G/f1pPZkC/e32WZLKx8FQ1DBEMS4nH/eKu3ZCuQGxy9YH38z0GxLNFoujq+jZLuzXLJxXPVsi3Hjq1Fz2Wiqul+o1JbKJsOiQdB0iNmB6UCxXufjL2EaWAnU7P08SkC7i2V9A58tzXdwQavEAADluYQ43vH5srIC2TCfMpFlp2o0vSwu60FRntUzuS89yxklFYadO+c9m9dpjQmfICJzzj+jBUP2KQ9n/Shidu6npA7umGc4KI9ocyD0+0jTVu7dUeba0z22iDhU4g8EIMsLlZrppGVwfHuUjf0NZe0R71WR8NZ4mD/8wLDtqNKi5FfP8hYxmlJU3ig9SlvJ0RafJ2NqyNo2a/jdykoAIjbzjnkjQpzVJevwJTp5LLQCUCpTwnnML1lc0djlBrznqocpVLEvvKQ8hDfl9GsVgtLcJq+xnmN+RPZAhlrUPfQrmh+zLAcXoPOggJpSK9FpRo/CpR9ykcB4+S8RAGQMfo3SCdXA4P8fmi882D0zCv8cQQL3AyqsLuikM43W6wTsN1eI5H8cdet1xCjUVJjWvVYSRHcMt7F0xCRJNzOvhNVBWXaUFZ0joSfG8dEa1EgM3ddRXFU/Nt2g7HIRZFA2UyQ1Bm5WOJDPe1knXvc1tMeavOc1+MnjP1KteTC+GA/BLzWkIaRfG0grRXsDlLtcUCeXZ7XkFUZESITB2LI8S1m0SW1xjejvxMH0LC+++urX1eXZwcX9Nl4NOD6w+/WvbWdMu29DVulx9vQTotGc2B/XIR0/jwUOK1qnHqnhnTdnViCDJeWnpF8+SLKh09d5SxG1U38bbJhrKleFwG0amEyCbPb8Q55CPgsKk5y2gd3urQN6S4PNE9XAbT2s8XSOv2wPG77oEjFxiIa9qwEAoIGVRg/40uL4zpuzP6cHEb/DZFbjpsqxQyEMwbq5K/n0wrqyPFXH2Qa/rbM1h++mjRupYj08bvVVQ5t9Tvi2hmmZqu/khtR3+yRsGLDPgMVeTvBemqSDSb5CXQm6exJVu5T+sHn0K7bgAPo7XIQtm2hFc6m8e4x7SQo5NvRcU+RiIO5qsc+mtG/3/T4gi/jgHi2Z9tzyRExJ5m4PFFeDg4OjlAeNoMmCHkmQXvXaqEPhrNYc0Dzn4pUdSmvoRFRzd0QEbgHLf/KN85OYVHUeGMCythnXMGwP5H8hS66/5j+OH89OJkcD0w2/Zs4GN7DdZLtUx9Mmw69VUBvnQKRIrAfMDpH5STBZHogt7oeKbwowQSqpsde3ubsLGbD6zsnVukKjO1Q81kAtZNUdkEhr2315nG7Hw2kMya7ckVUQTVNkhKB4+wJGA2GWOTFdEzzPeoBUiv69bmLMn2IlEt6An5wRpkWY8BZy9LeK6DSXnNtwbpjGdlz1HAH77B02Is4vh9oltAn8Q6kc+M5pbXnLp2Ptxd+gqCzsXtEWf3DJBTrshnQvRfmiSfTsj3v5bwy2BluOcKHPMqB7AvaNheOOBhfvFgN9cZZGZbeda3liD8Z3l9RLbBZH36MxCN2+Ds4/HZABkgz+ckbx50v+eTDjQz94LI518573cgKhpzbhW0neaEftjfHXUIJN1NrytfZ0AW5fM1uexM6hzxl3TdRQwYatarDHvPYfOnwLQFkQ9ga64UruKr22Wa/vy4ucojafY2PbbYmg9vTLbN3Xqmofdf6Rap9rPHZ3KN5Nfzk8PzT9f0yZCMto3UkEZKitnqIqtav72VZu50LB8zOCIaX7+f0UwRg6gtHEMz3wKFF5eDi4PLQXh2FH76S3h1fTk4OLWXziS4Rl+Yu6J8U3QJ0tH5p7ND29NoLpvGlV1qO/Nf2eH6z5C6W738dNZs9+j48oq/PLLW3p/h1Htgrq5W8VTNI/hLuV9lTRYIFjvf9jX3afgAwYhJex2s7f/qrTAjL8dnF5AWRqS9TFgFe8+E/W1Pu8Cppf9vC1Vzy1g+/CNt6XePM48gi+DyweW17SF2XT2Ue34ByatWrx5pFJ5zd9V/uP1X1NkaaAAWhnQgFoYBfRZJDjQM7b440o3/A/+XFUk=', 'engine09_retune.py': 'eNrFXGtv40ay/e5fwUxwl6RN07Ink03kZQBnLE+M9Wttzz6uIBC02JIYUyRDUh4/1vvb76mqbpKS5Yl3gcUNBhbZj+ru6nqcqm7m3bt3RyfHh1fW4L5QZTJXWW31fuhbVb4ox2o7z9IHaxIlpXWk4vOitupFlmRTK8pia5xnd6qcqmysrGgRJ7W/cYRepmucz6Mks2Z5GueLutq3xuiUxFGttiuVqnGd5BnTqWelqqjZ9jhKk5sykppSWXFS/ZonGQif5dZMMSFDtxrn6MbNxvm8WNQqthZZnaRWtDT5L0mWqdJKKktGVbG/8e7du41kXuRlbeXVxiQvrVsLJB37/PQiPPt8Gl7/cjk4OLyyPfv8YnD288nB1Urx6Z9PVkrwNvj7xeVSqdvfsPBfXvkqu0vKPBvejgJ71zaDj6s7z6oeKs/6tcozz5pF1Qwc8KwSfMnnnvWYFJMkVR6WOS2issJTXUZjdRONb/GI7dqYlPncKqKaOlqa7gVezRjZYl48WFFlZYUpKkAdBfhXxKaszsvxTIjxoyGVZZ1CP8tM+WSR8Q5GKdE5kkbVbaqiMvPnqi6TcWXalvk4jBbjkLcMa4HURFMVFqUaJxVoSMXGxung+pfzw6tgaEPYLsr8/vTkwh5tXA0GVNjzvtvz3n/X+6O3u/fe2+vtfT/a+NZiicNoUYrVYJ8x9CyPPWtRkZiSZERldJMq62YRT1VtOVU0h8SkCQl6VasCzC/zRRbjtyrSpK5cn8iKtKdlWImMRgsidpNkLJ1YWlaDAjMAQqbu8RZbN+ohZ4FWFviAMUqI5DZUBrKp6iisgp7f6+36Gx8Pzg6PDw+uB1fB04ZFyz24m9Jq+8M4GdeOzC9My+Deq1QJhtHzrt/z5gvQ8DAdIrXrWiS89yS8QyLde+/xCPxXnnu7o5EnYxiWvhgkLV+OMl+sjJKWzTCa/ojL54vl8na8gziarx1P2rZDRp1l1TJa5NVM1mnpuh69dJ8/rJTvus3Y/8inyf/D2M8bG7GawHBNVVU7kdsvVb0oM6PZfjWL9j5872SFX0Ez6DeqorKMHpzIi+uHQgUogmx9/53r+nV+81CrysHjTN1rmi4PAHsXks1wCi93+4UPKcf6/PltnJSOvFTBdblQnrpPqjrMb/nN3S/8L2VSq7CGyDpEwY9hICon95IsRq9gz4NE51/CLMqCI2iV8jBetEjroKpLV0ZP8ygO53m8SJVDpofsnNZ1+aGlLmCLN6BTwXKRXxVqHJK5CMm0hWk+ZpVy7EmaxFWo7oveXoht4KcfbA/DyiDu/nyVlsxBqBFdpyrcfRhUXVENq8LPoPCjYL6PR5q3Kn11jxno6c/dfb1Fc9m6Sqk4BAucilYlhtinQgfbgkJ3Hzu0rlgs5DzKFlEarqmpVB3CGIfk7qK4cnZdMG2iDet4EUd+UoXRXZSkZK8ct9+p6VDluWnKMmP2veEd1ZMBc+4i78GbFF7sUXtaxbfWoapVOU8yyEIytqaweAXZxxs4dxVh+cqqkmnGTxUZTxgs7T3rGTlbdqgWxiAHjt3yN4RIFTw9b7DSJDGbobuIfN6teiBxcSbFEOUjd18aEw+0NDlo4g1Hrh8VBUwnrwhNsSZMKhQfHzw99odiZB4tnkUJ9+2AiBMP76KR6+qxiRZkh9rogSDicygOzUXTounEmE017I1GGAVMuTJLhmlfpDDRpCpWNC7zqtLdqr5QhJVDVZbDZ3Ntg2B2OpgFTKEdTVWm54jRMMmfdvtmEnYYcv8wtNG2WWiXL1LSZY2jl0e8KZLxbTAc7WNU/MjqdQ9ecodNLXXNDObGCru4ilHKIiCgxZokVmfFZNEstux/2lvESBYsvImKuoA34zxWZKiMlRr2fxh59k0ytYnXlkMTJ+Ys/mcvCHqWgmEBIEzRlV2nQ+vTy+sYRb0UKnZX7SPz4GVbIrralBWNNo+2holZYIMpoB79Mkowo0tCkHM1KMu8dOzB6cX1P8Lrz2eD8Pwy/Hhwcvzz5cH18fmZLRRpkyHhRHDk/kG/gdpoPTkmBCrh0fHZp8HlxeXx2XV4/tfB5cnBRYfigyb4TfDU83afaaZSzJR16dcH+AjAehWeHl9dYSRNmlzOYq7JYAvcPznve1QBdApgBIDq2BwJhICxsMBXp+d/HthuQHhVtmsXYGL9yjqsCX8enB1/Oguvz89B4uAEqwXveuFueDG4/Dg4u6bpaItL6/QwHTFjALZJFuYZbLJ3DwumXTYEg+2Yp5HdeDL1NGCL1V0yVvIGPzIlAW/NN4nohgUzr9Jg7gMLOPcky4Ua7sIgVTUCkeDptn/nx4Bm45nj+uNiQX9TmoMggVvvjrSEifjcJWQ44RqdeqYRKGhazImYmOxHBR0P0+RWOXdLdJhCp2ulEEPF/0nH5FFVXeEfkiAn0iehHi33RlobJvB+tbv/RSXTGcABk9jhvywY4NU4KsgIOHP/49Gnoa1JCHtRZ4NtN1E9noXUq9uyLbXJtOqOjK6D4Ty6d3Y9ao35jlWSOvBDDs/33sX2F+5O29913c0OYQIHaaiKfDyrQFqWd7+yPpgNYPw5loRlwN8v5irV+1e0u8dtyAuSJYRLrkGYDV3Qg8uBCOx69HcPWPBHwLkff4SYZtOgdfbaQIcoZeHa+g7g8I9YrvhfzIQaIzz+baF4qXkGZKPgYZQzRCd/PMshrg4LNDGQ2PD+h+88ZgWZd69URYroTnCXXq3Y6c5y3WbI+0DEho02x3nO/bCZzYgApCM64u5L8YPuEVUAgBksJuxB20ELiiZK4vJ+TytZoOmIxykR5GBSYMxUOaKM7FxYTCvyTymcHD9plzNOjMdRtD8lMaVdk8TJvNlrVJXLGbuFHRXkR1PZLtRnM+Kw08kL2ChwutRL4nefohJHunVlwkMsANvSiH1asixbVvmaEDAIY0cIfmy+7/W2xsnmH3+UTmoSDIuOaekYFRbKlxMgVnE96U3L3q4yDcfJSOcUboJyvUS1urRGsEzJnzoax+Tu18rSzbIMPawTn5u3iU13O9jOhdMyYiwX1nmYgTsSoHBLkp7gyKdou3zQiAlMKPPiISRESsYekbrsonMPM/LwID3hz8RPwG91I16uFLr0Z6vnf9jkqTq85/MFNnuT7IdTbP/m+tVvC8RQ2De2jLJr3m+0KY9JsU54sN9uMwMCFsKNpJoAc0PWaVDjO49oXKDvC0ptaQ96dn4GVHB8Df99cgwvGZ6cX13ZLTd8yvh8icrYcfdbPpJQiKiL5hnE+Ea3JutY69ZcrcGGovCKZ0JNYEJp2OhuSs6LmCQNvrib0fBWIPsXLzIM0z7Hk1m6ehKNY3smlK7SOiJioAkK21xhSK1pCpEp5+LTHBr/0Y/zBcdNK5un59tGEdzdRyiz4NB6c9P/QGvJAEQ1KHje6AiSJAbbDI3XlStRxrYrJo8CQkt9o83t3EUIpwFPAEvbN8iBsoLsgTY7JVvO7jYXupvTdaLdJFcERBgae5vNu6GwBwoNU5gUT7Dt1zxtr+uxqcNXBEtO27KtF5KIpiixhoAgTcZJDS7fJJHOwWV3kipjftzk9cyk12Q9lc8U5rOgs/wds/zNTQfmdWsXTvtuFrQzMA322gYk5NG8CMkM9mRazdbwOvXDVkfzm0QQGYD5bMe5w1J/K6EL3VZ1RAbCZRenilXZc7qjNJLrvkUiW+loBFEbEApejPVoBuW4pXn7aVd9/xWbcniM2OITGZOrwSUew6vrAQcaMmDQzBolHTy0FextcljUuudNAVib36HlPMeU8nKNn9Y1r3lqU93x1QpayCJJRt2kmLVncPsCTNhmC6/f4BH0II4GSK6ncY9h+isMbod6m5E++Hx43LHRTQRizCWvm0sDlk0TzKRRJcOEcxVlelmYCL05YnHhrSXbEkpA1OFC+2jodUxh+2iSm4uCDluktpEZb5zP5wCpknTTCLiz+24bnvG+eZ06BvEkwl1M4m5qAKijOHOYA6WYYow7xGe0/TB+KAjWJD6bdk3QzmyhDOj+baDjBhRhB4gSyyb6sAeqWxaSNEcTuGOqHG7Dc2CfbyVy5ZLdkccJgUlnibW3vAVo+FNQm8SaHGPI6UTlPNReNeZMp8TpLdh/QI+vxeUHnz+Gl4O/fD6+HFyFP59f/yIB+uCqEwuzxESLMh/rJS0dmujBSZyL0rR45RylaVtQ90lR9j68TtJDdEZtKC3+wTULl+xWeEOZnDL/UkkO8Yq4UacP3eO1PklJFgOYAJDEFIjVSZRal+cfLazags3XB4ZcQyK3jwqqvilVdFtZdaLI/k9z+LVhKdEFg18MS5wuYaIhhwuEf/B7dJ6TAnPZfCjT1WC0k5HCdtkw2aMm9UMjrN2ds/Pwrwcnx4dhcyjT2RadVKLOHiU14V9u4sgq+872+hG9TjlvJ4pQ0px9khsZ9kbC53KRhRPSlbn3JS9vPdKbbrKjMpmOap7fsg7dg0ZEOkJGlqLkg4+/DNz9e5hjmIusAv/mDjca2vcUsD8E+u3BHu3H5iXGy6To0Lk8P7/eWc7AYwWqLEooYOVnxQMxpS69u8iD65j7nGoOE6xqTLrhxZRxbuZPbReZouRO8DvpaZMuQLBYl8MYf7DRj52s79w/PD89OD67op18/CagQbCtlJ6E6WK9YUsZ6NRjXbpem6g2pXeR68m2NO0wQco/NOlbU8EZRJi7ummpyGTkizqgXdoxabLmwLz3Y6gxYXipiKy9D5XMy7gK0GnH1i9N6e8f02xYOn6yKf1vf/WcQEycDZRvS3Te4lYtRQQ8AWOqoD12HEoVxZxyeNoJ1Y2oUqptJWIHEYG8pAsT+2nuk08BC6H8Dm2M+xyGT7Sv9Ctj4Gn81BDt9/biZ0p+85l1oDmyM7F5GugEys8+HUzZkqMLv9auqO398QLebrWVFOpW4+qORwQXqZnPvAb2ZyPSDrJc3pJtyvVpfsDHZqQ4FR9M+XSeIwdqbQyYU1LVMbbLfWG7dD3A7IRy5AHlHroVol8i5FQtT90WYhqoTp64jhPcscwIy3baVbhNQx0My74bxJK7+6zrjg0n9fl0EF5fHh+c2GT04CMX1UyfH2LCgEULxTR49YssTbJbZ55UBPfbk8YOY19r0uHxK01omDoZB2SvfejbJBxjDeTldW+RWyzmQZYloIVAjif5x/Vp5d/LKTNt2Uxx/zB58LNkbULOJwqqBUW2cxrQwkm2V1sWAHPm4Iwa8YGaRDqqDlawxZBajDx5XUokdJ0cOg3tJf/2ikczOPX689nx2afwdIDd/KjTCbnYTH0vJ2hsdtDhSGAYY9Q26FgFFlmSV0+kuyPanoiWtbn5ZJygvXXbv1tKpqu6STNormvB9JrMhs4dy/2MdYiVN1gP1oXL4nYNTGa8HfBeEv4DlFiPrOGjCQetwc2UvH5sosROjd3xUCIsZjpCXOLUap3gbkOgva5+a+WW7dHRNsCdFrFWj6CisX8IB35EmSfRbUrQrWg6n+Tf6yTyfntTgKpyGUSrOut4eP5nUXO7AGSDVMmmvBQ274Pr2U0EY5q9ibvo2TEiNAV1P1ZFbQ34h+97VZbq/7cEdAJvqWLbU6QgcBNF6WDDXt2DVZ7tL/Hr6OD4ZHAoPDOkVla3alsl6oyyB6du8ew3XZ9AwsS3TqTresU+Pvt4fnpxMqA0YePK+8CqAJIQHDwgOlOWrFZTsiW+pktwQRfX63FQCSZjHqrrPWd5pTLy4G/x7+KtN0iqFOnyf0omKx5tPpe/VNuUGVhgM/kulY43zP09c6UPQqOglQ962X6T2BAVYmT7Eje8GbLIKMN3jWC9GzF4IdABc1E0N1cCxl6eTnKGZOYbSajuuqd0r/kQIE7jQl5Gwbyst1Cp1VeJEJfSmk9nTMgdVHfD5kDY4L86Kumi3PKdLvrzYdTXKEqH4EzD/ROdC+tLctQzkCtWost60KW8iDTjqFMeGzVNsmoxmSRjzi7cqCyZZmEHoNvuMv4YL2pC+kzoZdpBaEN34UZ+ghOo6fwCjlYJYJCjTqf4g4PiYNfVyTkXcVG2VNVrq2pd4/yrcF9UTl5UtkQ3/m1WGAWA0WrWRoswyaEOW7i78MH4oKiuo/FtCOmP0jSoix05hK2LrUkmsQ13wlJ1DZ62alQ1uYTlTgX1KlCGhtTLAyfqzMOSQY9tyM2v/47VFsPQGm+ta90wuWkk1rxpIYB5JCvtBtpNk5Xo+0WQvtpwyccxb/T91ZcgjSBarT19Jo6Oz9Qo5sQ7MJy8cmi5JLuBuWlC7Si25HbcqhV0EaD2ioaWHhlvjbdaC41aTr2ow/pWIZZpvVJOWQttL/SvTIItTmgsMadesduh1nyOZBtxXZJRc5Vb7urpSJzbM3dDsVh8t6j2+YXlivKFgEKPIXlK0K3oNhC5GdkoOnSnHCU9IxxGJCwaLiUauYMgQAsoOh2D2A38mQzZYF5it6KhSTuy1MXQvltHWwsWz47CA/3ezpDAPi2vARraB3vQIy4XvPG347OzwSUDtNXUkNfRFW+d+tjatrAagP3RNMvp8p/BbRgJGtCRdhsIzujMd8uIRq4ZTqcl2VblNLkqiaKqJlvVpKn+3VzJcp5kwxLdD4bdSHspzG4P0tHJn6b5jdPAjU0BJJT7UxiXBJW1jWfsbsr1PGUe9dzlkhQrKZNxvwlM53VQbGIP/n4x+Hg9OLSeTLtnS/bryrM+nV9bTx1iz4TCIkhvKVfOnihfqU2lJAhpRvIkEzK3XTgZKlSeEcGbsTYshMty1aLJ30hAzE9sKyiF2lwSXOGVRkOGVX1twl/l9/4yon1aienQ18R0knanyJUPTrW99lYSGs+sav+0nkyDfvcuMolmY+gpBRsCa0uizH121y5IJ3w2OdnD6/kWDcbpIlb0XcA2C72V5tOKewtwbGMG3ZYy1aX6VWCmXjKdOE+CblKlcPfjiQ+wgu10ep5mJt1mtr2CDuXnvr5w0cwLDGD0wjvUQICJuU/TbjLNfIaBlr3pkrSwV21FRrvWjuS89K9kG1bKtAtd9rKdTeh0eWUnPOMB0VQ/opF2g1JG+j1acYRc00V1MhHtFqUfnmgZa7zbmx3bWp+20WKGDh54BQpsbparFtLVGDkX6Rva2jnaoz7ro9lZ2sF/5gVjd6OKS8DvP9xYntKSJvGp9FJagCfSpAXYsDb+mt03QpSC8rm84Jzz+6Q0S2mDFZo6WFwmKkCUwp1wBg9YNhdiQq0566nKvTjJG1QekA+5f2nFYrU0CKvtF5jfkL8soilrHfq9KTd8XyY4Se7BB2GhpPfXkhKNX2XqPqFy7FE8GcIAaKjexerkcriJz7fGbx6cZvMahwzxwk5GVdgdcQjf28Xs1FyjdDyKW/a62RgqLEoqXKsOIznPXF67zCQEqJzTKXqsqnGZFBQsrWPBt9YR8UoE2HwIoKLxzHzSt8NwxCJQUMYpsJmVTwQg7msl616Ot5jzVp3nvhg9Z+ZVrie360H5tc1rGWkUxdMK0t5n52DVFgvk2e3hD3GRJ0Js6lgcYa7dxLM8xvB25Gf6RBfef3dtBubq4uT4mm5Wnx5cf/zFsrdmW7al78S7/HgL1mnJaG4/LOdIjQ8PBbZVjVP3TJu2qoMhyHhp6RXNk8/VNIjuKGMXXDew2wRF2RIsl0Z0CCSyyf0bcQ75PD1sUtrSWqNcjYBDguexruG8k9Z+vo371mwXTQGQQQX23+gmyIQ+6qPvPbex3yaHTieXlUIggn4TVfJnp0lldaSYkxDyzcN2CstP33pa13L+3Pi9ihLzlvptEaWWSRnrLwB39PeABF6kwReoIn9mSfduonSal5jePBlbspT7pH7wCdp1cTyM1iILZdkCr3RQj3aPSSEnVW8Exz46AA7z9Rj9Laj/v0lxhF/H0PLsL7ZnwiPapabh8UV4ODg6OQBKbRqkQD5p8MGlfOujUSBWQEzdLwVMbeqLbXRgRqeNRM5x+4/y7ZhTeAQOU06zhHXO8QC7IYla6PuBH0KTqTSr92zMx/aaWS9lTPVpu6nU1y/4Ii8mUgTm81f/oJwuiFMX9EaHQIUfxRBUXezY29s0G7v5aM3euUXgkqodKiZLsK6Lyqaw7729Tjfe1RcNybrZnly7Bba2wVI6xYVBwZ6TTTYxEj3Dio9agvS6bmyOmWwvEg2DupA7rMGW9TPgIGZpnutoUnjze4104LOy5ijgjwnhcNEWcH6f+BbQB8VO5PNGc8lbjrA7nz0vfVlCdw3sEcf6TJAjr8hnRvRf6ySfo8jX05bsl5mV2T1X6JhXORd/RdH2wgE384sHu7kiIj3b4159EwzCf5bXR2QiTPCnP63RcxucfTo+GyAQ5P4c682D7jeSUoFi3r0g8vlX7lA4EBU9cy6VaTvNrYdhf3fUYZBUN7WufPECWZRPAuUCOalzxF8ndgcxZKhYjzLsvaTNn1vTEkQ+MFtzTXN1vrpcuumPt5vrURJtb9NjO1vzMZMJurla9zT8/ivdzG0OPORqzi/nJ4fnn6/pMyxpbRupIY2USLPVRVa1fnvTz9yTWT7gd0Q0vn7npekiBlFbOKZmvq8KLy4HFweXg/DsKPz8l/Dq+nJwcGovHZpzxr4w92/59u0SpaPzz2eHtqenuWwaV1ap7cx/ZYXrP+3qLvXy81mz3KPjyyv+mstaeyeJI/CBuQ5cjWdqHsFtyp01a7oAZux8L9ncUeLjBCMm7RW7tv6rN+2MvByfXUBaeCLtBc0q2Hsh7O972gXOLP3/qqiam9vyMSVpS797aHoEWcQuH1xe2x4g7OrR38tLXV61ep1LT+Hl7q76D7f/hqxbQw3EwpA+dw7DgD41JQcahnZfHOnG/wGsgIVi', 'engine09_main.py': 'eNqlW3tz3LiR/1+fgq5UHUkLombk9SaeCbdKa8uJ6mzZJyub5KamWBwS1DDikFyCI2ns+D77/RoNvkaU5Kur0oMEGo1Gd6PRDzDdlEVVW4USltrhz79UkQvra1omaSaFlRbCWodqnaUrAKy3dZoJK6yuy7BS1K1Ho9Pjnk1Yrw+SqthYJZ7QbiCsz9RhnvPtptxZobLysmkqwzxGA37KmMerKC13nqrDWjU4VrIOhXWjghMVbkph3YVKyUrVMs2DOAVoHkkz+CaTYZV7G1lXadQiqIooCLdRoKKiAvHhrazCaxmUlYxSlRY5dxwcfDy7+uund1/8hf1exqe31x8/fLYFPX+uivv25TQON+3LP4vrlF6WB1/OzmjoRPx0Il79NPmjmJ68EieTk5+XB+8+fTw9vyC8F++P/nbx5e9HF79OXwMBXq+Ki6Pz4opffi2uupe3X86O3p6/PTp/9+VkMv0Tprg8vToDFm8ymQpP/75eHhzEMrGyIoyDTRFvM+nk4UYKEoM7O7BUKSN/KC2P2gLiV0CyDrIiCmtwgQequnL0YHe+2R/IE/BQQuLQH3cO/TFdakE4lv5mTj0eUSUrT95jPkPcxp1Xst5WubVhytU6dBpi175ROQ+tJ69/dtwD6y6t11ZRylxDCbta2S7pSwJ4KykqK1pv8xsrza20lpWThZtVHM4Sr5Jh7EwnJz+9pD+uWNm2O1t72zIOa+noQcBuaFl7a3kfp9dS1ZhT02Xe0li5swZqSFxeegrcof+hCqsq3BG0iOtdKX00pnn980+u69XFaldL5eDxwSxabmEVrdNbyQu05D1YV8u4ewqqYpsDc9cA7Q4zRSyrijvlf/tu+GS2r/ffafke/5mxxK2vxC2SjfK/evQ/SzURlpWCCMmyV+4Ln56VrM07lh6mSlqX27xON/KsqorKsd9tyyyFzkjLUK43vbIJHW1cf5FryeQkFY2IZsk9iUUQmY59XG2x6QC6VR4ZHttddqRQOyiZjs79MVUqza8toIek0+ttsVXWA2yGkK3y6V3roXK+sk5Qx2KydM3izbhrrNnmZxuT21GxKTNZS5tm6sMkIRjbSIBAJ6N0nuflttYCq9CYw/gVtdUipbk1hoBlMsowcEnCLsXqmEEDW3dbMJkDZvY4yFRpfWCEwEWDenORIlj1CF+oF5v+Ru58p17Y6yKLi21tLwX0mBqUlLG9dAUeYV7XRdx1RSAppX1lM1uJeuBpZ1bExPppHptuVnXq3tP+R7h8G2ZpzLPMLPtQL0KvUM+7ABVLv0bDXZgDl//NWQslNiJyNX/0C5HZTMYWhVqqML+Wzv6O+26UBqSalhc+ox6l7295g+B4YxS35ZUhkagmm0uab1BqHvbF1wluoBbRulAyJ70g5TJwxEt6pee+kuTlVxjAqAB5+VYSLkgxfEQPFrOjn5aHjWYZpSD4fb3gtkY1+K3Rjj1NIEv1jDlRMmMp4LCQmd0goOUQatYQ3Rc0oAEtD/QENZwTWeNsMOvvBtTrSioiO4igLKtKH3bkOPyrwBL4NKu3OYRju09qmSq2VSSPijzbWWVV1EVUGCJboWIrG81zXmLd0LPlcypF+xYujqp9Ok1k7HTIBPGdTzSrnjlH2GpMaFCSO5OU1eQ1BNFrD7fwddC0vylh8AwzWUZGdXowL3yiYTCMWDkEL/Ikve7BmvcOUEGUdUAHpVQd3LB1lMd/T/NcVkYDIB8L+wVOZbRu9gdYwbZFH3Q4XklpnbTwfqXD9fzT0IyJMMuKu6BMI3iE/nvwWepzMNS7CMf+79u0gj0A40BhgHMbPpd+zMKVzNo37RjSW191GLrf0g7qN7ZjjWzMJPxiRmip4Bit/G83s3Bxw9qi3ZmGRjI5urUqMm0NmObhZC1isFejxGQ+0C5o0KGt517Od/2mAQVQDIB4EP0GJy/Jk+wROT/sEuzcpg2cx0S/b6XuNd30qA9KWfcAdtT9bSKm36mv51C98Me0RRO2fHIHYvAx091Xi0MbLzTYbdZCLzhgDEuYI5aWR58DRkDMAW3WqcHTDtwL8uAS6Bh8OL0y3QXPj7p2/NCaWvh6KklzOKCOhnM9qJ8zbkzO2pNGD7CaOZiYB9pOgg/FSsvdGRO8K9rmxoihZVQ13EY3EovdUwRRoGUaO8SUsFETQW+r5s31VPpVji7lC4nPKhBNZWHJhpCMPB+6+mggVGg3JyZ19g72Hz0x77RpGByUGpPb+e/0Ltiesl+tTYcDxpGPvAEdoR/ONysRrvyVpmczYqw2qx8yVbxsAmmNFC2+3bhP2JQftCJPGYzZQXsmUuBBYUcASxFmTkj2AyvEX/cJwsGrY7gPMLZ9E3tjAhKtFdhszr3IdXzBDHYmHs167/sTC2QYtXUoMvfKMnG8yclrgSFH94dTWN9pA52PQ7/5I6APpwRPYtQzt8c07yElwBVJJGhi/ZFoy8C1AZfZSuSv+OSccBNWTphekt5oVJhxXqa3Re3r18URrC65KUyqaZsu27Uz8UAG1azDhKJMPVzoTZRguQyhG11xw8uRYOKWQs0dfMya1lH66he/ntdlQ5zabpzyP5yd70+JpmSkY0Iddd7v+J9eTzLWM+3vjDiNaies6zC6CeA3wi75dXns1OVhksOXJ+2EG+EnaEvKwxptbVakBSxd4o9+Yh5NPIFV1KUAyUkpQF+dC9AClLz4cBunNRkxsYLvnkkBG0EodZBPPrd2EX0AzNlM+Aw3j3BELcY2xXIeqf2uxn7Paz2qv9XQptq2Fi7K/UgtIuLekrhaq0WtXw6syNecMt6tDyLh0/otpfyvWYZv/ot9N8p/zLsSAw+Kve54uymVM+pgCVL1AOxR/lW1la4gZFmwknl6nesTN8qpUS+v10ry0y6gP0h5OfVO1Iq6yiryH0l/NUCtazkdQyI24b3WGG8y7cG+fgb2NalkUfqL5bzTL7yodZrUeCB9paCGQjTeTRWEp116wmCTRWMniCws8wwy3WY15PqdjS/tcurVWTIyk6tQST/6txZsh8snONHXIyAqsi09+tPjhrlfZVUAPChyGagUhAVvXgfbEvG8Pz3Cgl6+dFpgfVRDc/xmEQuaw6Q06NH3KWenI3cz5M/TyWTShHQYumgCY5xDdpqrbZKkUSrzRrz92MV+5uRseWk1wTbNU8Bwgvw8dogvzBWTJHl2QpiaXuRYC7Z/N35ns6Oc7fVcW5PGYm5kSKuF2QMGeeu3RpH1o3bnWSHWqd+eOvJ2YSclNkDzcEgPdc7RJFgF9L/QRIdTeTQ9GT/mdJjWnSfWbVpkZh1A8gfrU5Vep3mYtekrVZOBwAg8UgiNtSqip6bO1upbZYE4dedZl+yZW/I+jGrN7ByOyVbHLd7BqESbhcec63jha4CWRA6hwnxHDLhZcnc/GABsCV8goT81uXQJsUQPWilY7uqIZ9RLH8S5vHt+eZxdl42ytNQcRwUgRiMw7WrTixa2Vvsn1KpbtWiR+7UYpxInSdWBBXpqyfZPaIIUt3FLt58DeR9JpXytKOY8s5dHepf3oPREe0DH+0AUW6Z5cF2E2R7on9lsJBRVpmkcRCnMAV58KPCwcZ1er/11Kl6+lLcuK9y7NLzOC1WnkUXJg5ml0s02Cxvv/9UJe//HncqSRsJGC8ozbApKFhVtpLCSUAupk4lhlSrWOSDptp3B6tCuU+ioFXwm8pJ6ne5+ExyneQTgKP8x4AMKo4NaBCLo24FXJ8YQSODq73cQIggVdSAO2+8iXISzPR4eqlWnHIYG6FJj6oJOwfa3ltgb9+qEU4qIF+DEEFVtPzZTIGlb+LSt0HcEh459RTZ5dZjiGN7GUBj/plFk9i/YCw8YuueVwengwXAUjao9PQCeSzuAI+nATGQsMw/3h9E4HUMN+GCaPTgc8W5DOKl8wxJtA/ts3DPir05+IWZ1i3hyMGTc3z2CXc9n4AduKsnt4US6T1sNqQdV8qlpxqD3J4nWlJWLyRsiJYPiqYfk9aQDBtJx9sJ3VMsSdxTLYPbHMWjF72SiNYxMmoyprmlkuSeNmo9UMi83ym8qo3QIwwM8sLRX1ewf4wAZ3blRulKSkjEySG+U1za5wsD1aqwGbKzqamaEZ8lcHcXeEMee9xRyZbd7SrmFbmKD4kcnHsHmwuZ+S+xvlN75Hvz+7ffv9qzlG/YaTr1MOrfi92Nyvly3zaqJW5NfGewzW5Af6DRuaNOI9fLB/Lse9HoyEW/o9zV+39Dva/y+WX7vBWFF2UVVQgunyVKkucOhYVNa906r6y0VjD7TW+UgVPXCmGqE3OzYR0d3RXVjizaHqSOEEbDCuDm2wFwhXGXftsfg1Ka4kbYItdr6tlb+AP6MBHTol56mi0YoUHPnU0XfCT0iAq/e5iZOqWJcAZs5reU9pBQUN0wZ/H7lyfw2rYp8Yb//cP7uS3D2j8+Tk+Dy06cre+mTO3HHWaLQ08TMHhvx5eOn/zzDEHtqH+hQtAfplUXpjECLC7jwQL/x+0VyG87y5MQWd8e2zMEmOTnxyh1WjO3ODQ/B/xR0LO3Gda2MIJd3jyF4E5BC5LI/vGnTgxEdY37lmwsDi9l02WMLB9+mb04BqvL1rYNRON0z5+KZf/Kg+9XkgLxUHWNyVWxlamKk1poM/aroVc+lX3Um0tyT+E5r7SHo35R4Btf3gbi7lNbGe3v69q9nnlYh6Nts42HnkHY5TVGswzlrEqQd3lkrPo+Kwgm5JBtxJ4g8Q7Uw1XSjnFYjPb8bGl5fV/KanBOMpJl0NqDDYIayAjbDzb5o3l3NXoFfU5QdlPobKGFE0BD1UE7kaP3B+o3S372Ke89bVzqsZA+58Q0RSeiSZq4ogtEKlubXnklStmkZYhsweGktN8TsB2kbG8xPMji0dZPhdLr0jYDRrg0KXmOLiAXaqxC96HH37enFu/N3FKQvGNFyES2fC2kbhvXqp4x7W7WhXVnRKWt/ujz/y/nF6Yfg9PLtX89/Owt+O/2g53tnC50EzGJK2W3VujFQRvmcfe3TnHWwV8GDGP4amYwgASdlpedSXk67toXfU1dQGfpNpcrgHitMzXc+gS7snb2cx+YZjuv83jzf4zkpW1TPETQ2x0Fzu+L+QUUnKR80xYOW/RrH/VP1jaZUE4UI32xWXlmlyQ4aG9bcrIsfLc3QaWhsVMMZznZWDGDSb32fiyJyM4JS/FBginq25FD18sCL5mLOKgtv5MmKc8Q6YaHvhzAUEPTu4gguRwVU3PCnJ4OrOcYvuCPdvl92+rGXb29IEWDgKCveEuXHg6X2CwYGa7Pbm+TQDrx/8zNlh54sH7Ub4s3PAsBHRC/dLvsvbHdKXhRpNbcwf0T5A9B53ArkoRHds6F8f0LchoIymh6XRFJsO8TZzk7QgsmgunM6tyiT1jedZEICqECghzlAsiP4GPCElysrVO3s1QPZLAk8NnUuLJ1crYr8r1u2fbqUdRt29S4EmfTMtS1BtDyofOHFXXIBNYUHqUNDp3cW0D2gZXOK9I82LpSxFexGrdzlnCtLGp2xkm3Vkdf2wuSygfuHykmf2WAg5tflmThNEni7rP9n3f2hyZ/mlnaR131bCJHSDuVM1Nsvv3Gqr3Nseyz9P/Fs1tZERxTfrG+6HFSXRVME/pERTZpe7BYY9ki9ao8zXfF3sI2M1T/98CFoLf+Xzx/Or74Ev51dnr8/b+y+Vnf3Jd9wg57vHQJ/sC4lwjOpUzS0cljy25SulzGbaX8W+5nDFHpfp3UqYcXgM0hYKNO087/ZDXjANwbtGV12bB0EKv4B8aDTnBLo65uNAcizx5Gwr6s0tmetI9o/cfs+2lLY7PvPjA0SNvuiAY/bn3bMY/3eLTigW4A+4IwP3uouRphZg0v2f4918n4jg2Ys3zSaD1B5HFSYCGM/qICAnDFwQq2vR/XOZb4yR3ekBgNaJ3Pc0FJcml+bZLqyzEUjiylvURlLPo5ZT9y7YjWEovsqQS3vceTA5jd9jyRp9aRlVdzKnALgXnZ2C+gok2FuUUxmkc8rK9qIrbbqA0CXAYYE3FU4zpmCXkWqgREw+XjyT9wfOTQ6bXvgeO8pnXF1jc4BN4Z+TeHetBge87/3EA0wCPyOedqMXJhIpZn8pyfc2RbRU+7s6Ob6cXeWd9BDF9b454aMGR82uhggd0tahH6g2GJbP7LVLk1ZbA6QR/ZOUaqmEtcW4ZSuwrXxwX5oABbc+rbOocoYanhvm6t9i5Ol7/flwmEL2S/EyRgWTsVqKqKp34suxC0C7VId+uFUU3Hor6aGjEM/mu7RgWX36RhFaOZ7Bm2c+GXsvYNT/b4KNwjDSngxcQK3MIjUrQOOHdsFeBnW+pJdoY0v3FIPvbbeD/eNzz7AQ5O5AyxdTrkrs2p6n0PF5A6R6URuwFm9B8MPLL10P04WWElTofYNQxA46NREH51uCfbXOYbZfEfh76dNRZd01TEiLxDPfP+Q/pcV/e+Kyf0X+vThQQFn2Ka3BF3NigvKkvm8imts33IFj7+5ZCra+6hiUC4WoQr6S1mYlSw5fwqha7QDtnBTAEO7CavdGDtMl28Gj1AzKFl3k8KgaaiQeKRquhw7b+cpsu0mJx4HtkdXUp0btyv87UEt22Hk5de8RmeoLVxWeH41bXWOmbtgFpvaXb94t5y3z8+uWS+1BQ/0ZV3lO3xFGGtHuAUPhYmj9k58ORco0MuDmmIcaYMuyDn2eKWO0G43tus+wZMGSRAVfPthjCFlmFZmG5lrHwPDBoNJAPRnQX9GWeXpyo3DiBe95RkGDJS0Eb1tdi3FN9lWPrPfmj322N6CbmnynlasPogx/KyuXG7+/8w/0yxibPD1Icw6tJf+sNWYp6Nha/+AWbJEnpIqQwYEBmXj5Fegi/Wj8u3b2UX97292932GPbP7+d3vmiF72S0WjuMuD0cGt7nd3tDOk2iHDhew3gEF/FYQBJrNjXDo6GYFzyIeNUGsc9+aywWz7isK6FiR1HchZe9JWhTH9Fz7Nhjh7WXP2lyY8fpbzg8gsAZA4G+gg3I6L8y3L023ucdujuDB4H2r/WA7tN8KtMP3j6UeIsB0luUp6BYKI24p+ZRiAIfi5JgaqIfxoE3Fw26KLsUawI8P+pcQ7JmudNidQ27PGr8ZIRFL98FnSY843AwzcLcZwfnF1dnl58uzq9Or808XXn1fD3HYVzr8N15+d3uAL6pYqaIYlpJB8Did8Ustc2sT7iz4xLBA64Ii92Kjsb3/fOl61vvmdjLfRyCUFEc1VxFyCd+V4otEIvaKpBnw6sRqK+w0JESMkCuEybdgT3P5gWphGhkFWJDSziqSQcLjlWd9wmKMBSY80NOQNgaH6LTNeOdbcXvBQs3ZHbLqcJVJio0olQK+AHxbHXXZY4qRPeuMzkr+EodT6Bbm712taBYsGa6o8XedEjfSiBp0IsWSSaK/pa0oQMMq6KeE4x9mFNpa7zGzxYbfuDP00a0uqeTYqNjoxLa7o0zeyszC1ocINFf5UhR9HsshJ6REJV3ijolKhXW3TqO1FiLRSeL3rF/DjFQy7lKBFheotUIAJclq3l21OD79rEcDfVbsNPdBF+iHHfGsczJA/TQmpxU5b6s/awvLelv16YseSZdkIY1LtllmJTLUgyg35llXzB2T3oHUejI2G5KuhFgXRcPITtuTtCYL4HGy+TyPsm2M+TEHjFOvXtCyTH8jrL/F2mJroEGvhw2I6D5CMteL9Uq0604JNspBm5L83bEzFn99+fT+6u+nl2cBIsKr4NPFh396CEDtB4WdJ2I3PcB95JtOnl3Yd7Zou84/B+/O3n+gGocg5QXdSmuT/7r7/JOTgvyl2AL2RfxQqmbZVtkS7Y/yN0qEx6uus2Ll2C/herUfFiReqvRnxYjVvrKtchJBn+Nm2vIEdeHcPfjAbTEoworx0qoYS0D1GrVPjiamuJn87lh/1oxjMpbH7QeCJml49o/PZ5fnH88urjD+7aePnz+cXZ3Z4oFtdvVn0cz5YdoQSw70h5VBAEcw0EQEgT3j0v7B/wJrejNu'}
for filename, encoded in PAYLOADS.items():
    (WORK/filename).write_bytes(zlib.decompress(base64.b64decode(encoded)))
    print('WROTE', filename)

In [ ]:
# Resolve exact inputs, without silently choosing among ambiguous archives/cache pairs.
def only(paths,label):
    paths=list(dict.fromkeys(p.resolve() for p in paths))
    if len(paths)>1: raise RuntimeError(f'Multiple {label}: {paths}. Set explicit path in cell 1.')
    return paths[0] if paths else None
original=None
if not SMOKE_TEST:
    original=Path(ORIGINAL_ZIP) if ORIGINAL_ZIP else only(INPUT.rglob('FLIDS_Experiment08_Results.zip'),'original result archives')
    if original is None or not original.is_file():raise FileNotFoundError('Attach FLIDS_Experiment08_Results.zip using Kaggle Add Input.')
    pair_names=['flids_exp02_cache.npz','flids_exp02_fingerprints.npy']
    if CACHE_DIR:
        source=Path(CACHE_DIR)
        if not all((source/n).is_file() for n in pair_names):raise FileNotFoundError('CACHE_DIR must contain both cache files')
    else:
        source=only((p.parent for p in INPUT.rglob(pair_names[0]) if (p.parent/pair_names[1]).is_file()),'original cache pairs')
    if source:
        for n in pair_names:
            if not (WORK/n).exists():shutil.copy2(source/n,WORK/n)
    if NF_UQ_DATA:os.environ['NF_UQ_DATA']=NF_UQ_DATA
    if RESUME and not (WORK/'FLIDS_Experiment09_FedProx_Retune'/'records').exists():
        rootname='FLIDS_Experiment09_FedProx_Retune'
        prev=only((p.parent for p in INPUT.rglob('resume_identity.json') if p.parent.name==rootname),'resume folders')
        archives=list(INPUT.rglob('FLIDS_Experiment09_Results.zip'))
        if prev:
            shutil.copytree(prev,WORK/rootname,dirs_exist_ok=True)
        elif archives:
            resumezip=only(archives,'Experiment 09 resume archives')
            with zipfile.ZipFile(resumezip) as z:
                for item in z.infolist():
                    parts=Path(item.filename).parts
                    if not parts or parts[0]!=rootname or item.is_dir() or '..' in parts:continue
                    dst=WORK.joinpath(*parts);dst.parent.mkdir(parents=True,exist_ok=True)
                    with z.open(item) as src,dst.open('wb') as dest:shutil.copyfileobj(src,dest)
        print('Resumed trial records:',len(list((WORK/rootname/'records').glob('trial__*.json'))))
print('Original ZIP:',original)

In [ ]:
# Run preparation, split verification, FedProx retune, and threshold audit in one subprocess.
cmd=[sys.executable,'-X','faulthandler',str(WORK/'engine09_main.py'),'--work',str(WORK)]
if SMOKE_TEST:cmd+=['--smoke']
else:cmd+=['--original',str(original)]
env=os.environ.copy();env['PYTHONUNBUFFERED']='1'
logpath=WORK/'experiment09.log'
with logpath.open('a',encoding='utf-8') as log:
    proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
    for line in proc.stdout:
        print(line,end='',flush=True);log.write(line);log.flush()
    rc=proc.wait()
if rc:raise RuntimeError(f'Experiment 09 failed (exit {rc}); see {logpath}. Completed candidates remain available for resume.')

In [ ]:
import pandas as pd
out=WORK/'FLIDS_Experiment09_Results'
result=WORK/('FLIDS_Experiment09_SOFTWARE_TEST_ONLY.zip' if SMOKE_TEST else 'FLIDS_Experiment09_Results.zip')
with zipfile.ZipFile(result) as z:
    assert z.testzip() is None
    status=json.loads(z.read('FLIDS_Experiment09_Results/run_status.json'))
print(json.dumps(status,indent=2))
if not SMOKE_TEST:
    assert status['new_training_trials']==80 and status['final_models']==80 and status['final_operating_points']==240
summary=pd.read_csv(out/'equal_domain_summary.csv')
display(summary[summary.target_fpr==.01])
display(pd.read_csv(out/'fpr_goal_coverage.csv'))
final_download=BASE/result.name
if final_download.resolve()!=result.resolve():shutil.copy2(result,final_download)
print('Download ZIP:',final_download,'| MB:',round(final_download.stat().st_size/1048576,2))
display(FileLink(str(final_download)))

## Interpretation
- The four-candidate grid is a limited retune, not evidence of global optimality. Historical FedProx received additional search effort; retain the original results and document the revised grid.
- All selection is source-only, with the same original splits and score policy. The target domain is never used for tuning or threshold selection.
- Five seeds nested within each of four domains are not 20 independent networks. Flow-level binomial intervals are descriptive under an independence assumption.
- Score storage is float64, while neural inference remains float32. Float32 round-trip diagnostics cannot repair Experiment 03 archives.
- The original threshold is conservative and may undershoot the nominal source FPR. Class-balanced sampling changes precision/AP; exact row fingerprints do not establish capture-level independence.